# Experimentos

In [ ]:
RUN_NUTS = False    # ← flip to True to execute

In [ ]:
import json
import warnings
from pathlib import Path

import arviz as az
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
from imblearn.over_sampling import SMOTE
from scipy.special import expit
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    brier_score_loss,
    classification_report,
    fbeta_score,
    make_scorer,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    StratifiedGroupKFold,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

DATA_DIR   = Path("/kaggle/input/datasets/josepablosantos/cambiar-preprocessing/processed")
OUTPUT_DIR = Path("outputs/experiments")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
rng  = np.random.default_rng(SEED)

print(f"PyMC  {pm.__version__}")
print(f"ArviZ {az.__version__}")

## Cargar datos + reportar features usadas

In [ ]:
# %% [1] ── Load data + feature manifest ─────────────────────────────────────
df = pd.read_parquet(DATA_DIR / "dataset_student_dropout_imputado_investigacion.parquet")
school_lookup = pd.read_csv(DATA_DIR / "school_code_lookup.csv")

with open(DATA_DIR / "feature_manifest.json", encoding="utf-8") as f:
    M = json.load(f)

# Convenience aliases
TARGET      = M["target"]
NUM_COMMON  = M["numeric_common"]
NUM_TEC21   = M["numeric_tec21_model"]   # was: M["numeric_tec21"]
CAT_COMMON  = M["categorical_common"]
FLAGS       = M["missing_flags"]
ERA_COL     = "era_code"                           # 0=Pre-Tec21, 1=Tec21
SCHOOL_COL  = "school_code"
N_SCHOOLS   = df[SCHOOL_COL].nunique()
N_ERAS      = 2

# Drop rows where target is missing (retention=NaN propagates to dropout=NaN)
df_model = df.dropna(subset=[TARGET]).reset_index(drop=True)

print(f"Total usable rows  : {len(df_model):,}")
print(f"Dropout rate       : {df_model[TARGET].mean():.3%}")
print(f"Schools            : {N_SCHOOLS}")
print(f"Era distribution   :")
print(df_model.groupby("era")[TARGET].agg(["count", "mean"]).rename(
    columns={"count": "n", "mean": "dropout_rate"}))

## Train/Test split

Stratify jointly by era × dropout to preserve marginal rates in both splits.

In [ ]:
df_model["_strat"] = (
    df_model[ERA_COL].astype(str) + "_" + df_model[TARGET].astype(int).astype(str)
)

X_raw, X_test_raw, y_train, y_test = train_test_split(
    df_model.drop(columns=[TARGET]),
    df_model[TARGET].values,
    test_size=0.20,
    stratify=df_model["_strat"],
    random_state=SEED,
)
X_raw   = X_raw.reset_index(drop=True)
X_test_raw = X_test_raw.reset_index(drop=True)

# Group arrays used by PyMC (extracted BEFORE sklearn transforms)
school_train = X_raw[SCHOOL_COL].values.astype(int)
school_test  = X_test_raw[SCHOOL_COL].values.astype(int)
era_train    = X_raw[ERA_COL].values.astype(int)
era_test     = X_test_raw[ERA_COL].values.astype(int)

print(f"\nTrain : {len(y_train):,} rows  |  dropout {y_train.mean():.3%}")
print(f"Test  : {len(y_test):,}  rows  |  dropout {y_test.mean():.3%}")


## Column transform
Strategy:
 - StandardScaler on all numeric columns (NaN already imputed in preprocessing)
 - OneHotEncoder on categoricals — handles 'No information' / 'Does not apply' as legitimate categories (handle_unknown='infrequent_if_exist')
 - Missing flags passed through as-is (already 0/1)

In [ ]:
numeric_cols_m1  = NUM_COMMON                   # Model 1 & 2: common only
numeric_cols_m3  = NUM_COMMON + NUM_TEC21        # Model 3: add tec21-only filled
flag_cols        = FLAGS

def build_preprocessor(numeric_cols):
    return ColumnTransformer(
        transformers=[
            ("num",  StandardScaler(),                numeric_cols),
            ("cat",  OneHotEncoder(
                        handle_unknown="infrequent_if_exist",
                        sparse_output=False,
                        min_frequency=30,          # collapse rare categories
                    ),                             CAT_COMMON),
            ("flags", "passthrough",               flag_cols),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

preprocessor_common = build_preprocessor(numeric_cols_m1)

# Fit once on train, reuse for Models 1 & 2
X_train_sk = preprocessor_common.fit_transform(X_raw)
X_test_sk  = preprocessor_common.transform(X_test_raw)

feature_names = preprocessor_common.get_feature_names_out()
print(f"sklearn feature matrix: {X_train_sk.shape[1]} columns")
print(f"  numeric  : {len(numeric_cols_m1)}")
print(f"  one-hot  : {X_train_sk.shape[1] - len(numeric_cols_m1) - len(flag_cols)}")
print(f"  flags    : {len(flag_cols)}")


## Evaluation Helper

In [ ]:
def find_recall_threshold(y_true, y_prob, target_recall=0.80):
    """Return the highest threshold that still achieves target_recall.

    precision_recall_curve returns thresholds in increasing order with recall
    in decreasing order. We iterate forward keeping the last (highest) t where
    recall >= target_recall, which maximises precision at the given recall target.
    """
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    best_t, best_p, best_r = thresholds[0], precision[0], recall[0]
    for p, r, t in zip(precision[:-1], recall[:-1], thresholds):
        if r >= target_recall:
            best_t, best_p, best_r = t, p, r
    return best_t, best_p, best_r


def evaluate(name, y_true, y_prob, threshold=None, target_recall=0.75):
    """Evaluate model; auto-selects threshold for recall >= target_recall."""
    if threshold is None:
        threshold, thr_prec, thr_rec = find_recall_threshold(
            y_true, y_prob, target_recall
        )
    else:
        thr_prec = precision_score(
            y_true, (y_prob >= threshold).astype(int), pos_label=1, zero_division=0
        )
        thr_rec = recall_score(
            y_true, (y_prob >= threshold).astype(int), pos_label=1
        )

    auc   = roc_auc_score(y_true, y_prob)
    ap    = average_precision_score(y_true, y_prob)
    brier = brier_score_loss(y_true, y_prob)

    print(f"\n{'─'*55}")
    print(f"  {name}")
    print(f"{'─'*55}")
    print(f"  ROC-AUC         : {auc:.4f}")
    print(f"  PR-AUC          : {ap:.4f}")
    print(f"  Brier           : {brier:.4f}")
    print(f"  Threshold used  : {threshold:.3f}  (auto for recall≥{target_recall})")
    print(f"  Dropout recall  : {thr_rec:.3f}")
    print(f"  Dropout prec    : {thr_prec:.3f}")
    print(classification_report(
        y_true, (y_prob >= threshold).astype(int),
        target_names=["retained", "dropout"],
    ))
    return {
        "model":             name,
        "roc_auc":           auc,
        "pr_auc":            ap,
        "brier":             brier,
        "threshold":         threshold,
        "dropout_recall":    thr_rec,
        "dropout_precision": thr_prec,
    }

results = []

# Experimentos (Baseline + Fixed Effects + Multilevel)

## Baseline: Regresión logística (Sin Tec21 o Escuela/Departamento 

In [ ]:
lr_baseline = LogisticRegression(
    C=1.0,
    solver="saga",
    penalty="l2",
    max_iter=2000,
    class_weight="balanced",   # compensates class imbalance
    random_state=SEED,
    n_jobs=-1,
)
lr_baseline.fit(X_train_sk, y_train)

prob_train_m1 = lr_baseline.predict_proba(X_train_sk)[:, 1]
prob_test_m1  = lr_baseline.predict_proba(X_test_sk)[:, 1]

results.append(evaluate("M1 – Baseline LR (no groups)", y_test, prob_test_m1))

# Top absolute coefficients
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef":    lr_baseline.coef_[0],
}).sort_values("coef", key=abs, ascending=False)

print("\nTop 15 features by |coef|:")
display(coef_df.head(15))

In [ ]:
# Priority 3 — Cross-validation with F2 scoring (recall-weighted)
f2_scorer = make_scorer(fbeta_score, beta=2, pos_label=1)
cv_strat  = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_scores_m1 = cross_val_score(
    lr_baseline, X_train_sk, y_train,
    cv=cv_strat, scoring=f2_scorer, n_jobs=-1,
)
print(f"M1 – Baseline LR  |  F2 CV (5-fold): {cv_scores_m1.mean():.3f} ± {cv_scores_m1.std():.3f}")
print(f"  per-fold: {np.round(cv_scores_m1, 3)}")

## Fixed Effects

In [ ]:
X_train_m2 = np.column_stack([X_train_sk, era_train])
X_test_m2  = np.column_stack([X_test_sk,  era_test])

lr_era = LogisticRegression(
    C=1.0,
    solver="saga",
    penalty="l2",
    max_iter=2000,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
)
lr_era.fit(X_train_m2, y_train)

prob_test_m2a = lr_era.predict_proba(X_test_m2)[:, 1]
results.append(evaluate("M2a – LR + era fixed effect (sklearn)", y_test, prob_test_m2a))

era_coef = lr_era.coef_[0, -1]
print(f"\nCoefficient for era_code (Tec21 vs Pre-Tec21): {era_coef:.4f}")
print(f"  → Tec21 log-odds shift: {era_coef:+.4f}  (OR = {np.exp(era_coef):.3f})")


# %% [6b] ── Model 2b: PyMC — separate intercepts per era (partial pooling) ───
# With only 2 groups, partial pooling ≈ shrinkage toward shared mean.
# This is the foundation for the full multilevel model.
#
# Model:
#   logit(p_i) = α[era_i] + X_i @ β
#   α[k] ~ Normal(μ_α, σ_α)    k ∈ {0=Pre-Tec21, 1=Tec21}
#   μ_α  ~ Normal(0, 1.5)
#   σ_α  ~ HalfNormal(1)
#   β    ~ Normal(0, 1)         (weakly informative on standardised features)

coords_era = {
    "era":     ["Pre-Tec21", "Tec21"],
    "feature": list(feature_names),
}

X_train_pt = X_train_sk.astype("float32")
X_test_pt  = X_test_sk.astype("float32")

with pm.Model(coords=coords_era) as era_model:
    # Data containers — allows out-of-sample prediction without re-compiling
    X_data    = pm.Data("X",        X_train_pt,  dims=("obs", "feature"))
    era_data  = pm.Data("era_idx",  era_train,   dims="obs")

    # Hyperpriors for era intercepts
    mu_alpha  = pm.Normal("mu_alpha", 0.0, 1.5)
    sigma_alpha = pm.HalfNormal("sigma_alpha", 1.0)

    # Era-level intercepts (non-centered)
    z_era   = pm.Normal("z_era", 0.0, 1.0, dims="era")
    alpha   = pm.Deterministic("alpha", mu_alpha + sigma_alpha * z_era, dims="era")

    # Feature coefficients
    beta    = pm.Normal("beta", 0.0, 1.0, dims="feature")

    # Linear predictor
    logit_p = alpha[era_data] + pm.math.dot(X_data, beta)

    # Likelihood
    y_obs   = pm.Bernoulli("y_obs", logit_p=logit_p, observed=y_train)

In [ ]:
with era_model:
    map_era = pm.find_MAP(progressbar=True)
    logit_p_test_era = (
        map_era["alpha"][era_test]
        + X_test_pt @ map_era["beta"]
    )
    prob_test_m2b_map = expit(logit_p_test_era)

results.append(evaluate("M2b – PyMC era partial pooling (MAP)", y_test, prob_test_m2b_map))

## NUTS

Opcional (TOMA HORAS)

In [ ]:
if RUN_NUTS:
    with era_model:
        approx_era = pm.fit(
            n=30_000,
            method="advi",
            random_seed=SEED,
            progressbar=True,
        )
        trace_era = approx_era.sample(2000, random_seed=SEED)
    
    # Posterior predictive on test set
    with era_model:
        pm.set_data({"X": X_test_pt, "era_idx": era_test})
        ppc_era = pm.sample_posterior_predictive(
            trace_era, var_names=["y_obs"], random_seed=SEED, progressbar=False
        )
    
    prob_test_m2b = ppc_era.posterior_predictive["y_obs"].mean(dim=["chain", "draw"]).values
    results.append(evaluate("M2b – PyMC era partial pooling (ADVI)", y_test, prob_test_m2b))
    
    print("\nEra intercepts (ADVI posterior mean):")
    alpha_post = trace_era.posterior["alpha"].mean(dim=["chain", "draw"]).values
    for k, name in enumerate(["Pre-Tec21", "Tec21"]):
        print(f"  α[{name}] = {alpha_post[k]:.3f}  (OR = {np.exp(alpha_post[k]):.3f})")

## Modelo Completo: Regresión Logística Multinivel

### Preprocesar para Multinivel

In [ ]:
preprocessor_m3  = build_preprocessor(NUM_COMMON + NUM_TEC21)
X_train_m3 = preprocessor_m3.fit_transform(X_raw).astype("float32")
X_test_m3  = preprocessor_m3.transform(X_test_raw).astype("float32")
feature_names_m3 = preprocessor_m3.get_feature_names_out()

print(f"Model 3 feature matrix: {X_train_m3.shape[1]} columns")
print(f"  common numeric   : {len(NUM_COMMON)}")
print(f"  tec21-only model : {len(NUM_TEC21)}")

### Definir modelo

In [ ]:
n_features_m3 = X_train_m3.shape[1]

coords_full = {
    "school":   school_lookup["school"].tolist(),
    "feature":  list(feature_names_m3),
}

with pm.Model(coords=coords_full) as multilevel_model:
    # ── Data containers ──────────────────────────────────────────────────────
    X_data_m3    = pm.Data("X",          X_train_m3,    dims=("obs", "feature"))
    era_data_m3  = pm.Data("era_idx",    era_train,      dims="obs")
    school_data  = pm.Data("school_idx", school_train,   dims="obs")

    # ── Level-2 hyperpriors ──────────────────────────────────────────────────
    mu_alpha    = pm.Normal("mu_alpha",   0.0, 1.5)
    sigma_alpha = pm.HalfNormal("sigma_alpha", 1.0)

    # ── School random intercepts (non-centered) ──────────────────────────────
    z_school    = pm.Normal("z_school", 0.0, 1.0, dims="school")
    alpha_school = pm.Deterministic(
        "alpha_school",
        mu_alpha + sigma_alpha * z_school,
        dims="school",
    )

    # ── Era fixed effect ─────────────────────────────────────────────────────
    beta_era    = pm.Normal("beta_era", 0.0, 1.0)

    # ── Student-level feature coefficients ───────────────────────────────────
    beta        = pm.Normal("beta", 0.0, 1.0, dims="feature")

    # ── Linear predictor ─────────────────────────────────────────────────────
    logit_p = (
        alpha_school[school_data]
        + beta_era * era_data_m3
        + pm.math.dot(X_data_m3, beta)
    )

    # ── Likelihood ───────────────────────────────────────────────────────────
    y_obs = pm.Bernoulli("y_obs", logit_p=logit_p, observed=y_train)


### Entrenar modelo

In [ ]:
# with multilevel_model:
#     map_full = pm.find_MAP(progressbar=True)
#
# logit_p_test_map = (
#     map_full["alpha_school"][school_test]
#     + map_full["beta_era"] * era_test
#     + X_test_m3 @ map_full["beta"]
# )
# prob_test_m3_map = expit(logit_p_test_map)
# results.append(evaluate("M3 – Multilevel school+era (MAP)", y_test, prob_test_m3_map))

# ── Step 2: ADVI (~minutes) ──────────────────────────────────────────────────
with multilevel_model:
    approx_full = pm.fit(
        n=5_000,
        method="advi",
        random_seed=SEED,
        progressbar=True,
    )
    trace_full = approx_full.sample(500, random_seed=SEED)

# ── Step 3: Predict from posterior means (fast) ─────────────────────────────
beta_mean         = trace_full.posterior["beta"].mean(dim=["chain", "draw"]).values
beta_era_mean     = float(trace_full.posterior["beta_era"].mean())
alpha_school_mean = trace_full.posterior["alpha_school"].mean(dim=["chain", "draw"]).values

logit_p_test_m3 = (
    alpha_school_mean[school_test]
    + beta_era_mean * era_test
    + X_test_m3 @ beta_mean
)
prob_test_m3 = expit(logit_p_test_m3)
results.append(evaluate("M3 – Multilevel school+era (ADVI)", y_test, prob_test_m3))

## M4 – Gradient Boosting: XGBoost & LightGBM (Priority 2)

In [ ]:
# Priority 2 — XGBoost with scale_pos_weight
class_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Class ratio (neg/pos): {class_ratio:.2f}  → scale_pos_weight = {class_ratio:.1f}")

xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=class_ratio,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="aucpr",
    early_stopping_rounds=30,
    random_state=SEED,
    n_jobs=-1,
)
xgb_model.fit(
    X_train_sk, y_train,
    eval_set=[(X_test_sk, y_test)],
    verbose=False,
)
prob_test_m4 = xgb_model.predict_proba(X_test_sk)[:, 1]
results.append(evaluate("M4 – XGBoost (scale_pos_weight)", y_test, prob_test_m4))

# F2 cross-validation — n_jobs=1 inside model to avoid nested parallelism with cv n_jobs=-1
xgb_cv = XGBClassifier(
    n_estimators=100, learning_rate=0.05, max_depth=6,
    scale_pos_weight=class_ratio, random_state=SEED, n_jobs=1,
)
cv_scores_m4 = cross_val_score(
    xgb_cv, X_train_sk, y_train, cv=cv_strat, scoring=f2_scorer, n_jobs=-1,
)
print(f"\nM4 – XGBoost  |  F2 CV (5-fold): {cv_scores_m4.mean():.3f} ± {cv_scores_m4.std():.3f}")

In [ ]:
# Priority 2b — LightGBM with is_unbalance=True
lgbm_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    is_unbalance=True,
    subsample=0.8,
    bagging_freq=1,        # required for subsample to take effect in LightGBM
    colsample_bytree=0.8,
    random_state=SEED,
    n_jobs=-1,
    verbose=-1,
)
lgbm_model.fit(X_train_sk, y_train)

prob_test_m4b = lgbm_model.predict_proba(X_test_sk)[:, 1]
results.append(evaluate("M4b – LightGBM (is_unbalance)", y_test, prob_test_m4b))

# F2 cross-validation — n_jobs=1 inside model to avoid nested parallelism with cv n_jobs=-1
lgbm_cv = lgb.LGBMClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=6,
    is_unbalance=True, subsample=0.8, bagging_freq=1,
    random_state=SEED, n_jobs=1, verbose=-1,
)
cv_scores_m4b = cross_val_score(
    lgbm_cv, X_train_sk, y_train, cv=cv_strat, scoring=f2_scorer, n_jobs=-1,
)
print(f"\nM4b – LightGBM  |  F2 CV (5-fold): {cv_scores_m4b.mean():.3f} ± {cv_scores_m4b.std():.3f}")

## M5 – SMOTE Oversampling (Priority 4)

In [ ]:
# Priority 4 — SMOTE oversampling
smote = SMOTE(sampling_strategy=0.3, random_state=SEED)
X_resampled, y_resampled = smote.fit_resample(X_train_sk, y_train)
print(f"After SMOTE: {y_resampled.sum():,} positive / {len(y_resampled):,} total "
      f"({(y_resampled==1).mean():.1%})")

# M5 – SMOTE + Logistic Regression
lr_smote = LogisticRegression(
    C=1.0, solver="saga", penalty="l2",
    max_iter=2000, class_weight="balanced",
    random_state=SEED, n_jobs=-1,
)
lr_smote.fit(X_resampled, y_resampled)
prob_test_m5 = lr_smote.predict_proba(X_test_sk)[:, 1]
results.append(evaluate("M5 – SMOTE + LR (balanced)", y_test, prob_test_m5))

# M5b – SMOTE + XGBoost (no scale_pos_weight — imbalance handled by SMOTE)
xgb_smote = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="aucpr",
    early_stopping_rounds=30,
    random_state=SEED,
    n_jobs=-1,
)
xgb_smote.fit(
    X_resampled, y_resampled,
    eval_set=[(X_test_sk, y_test)],
    verbose=False,
)
prob_test_m5b = xgb_smote.predict_proba(X_test_sk)[:, 1]
results.append(evaluate("M5b – SMOTE + XGBoost", y_test, prob_test_m5b))

## M6 – Era Interaction Features + XGBoost (Priority 5)

In [ ]:
# Priority 5 — Era interaction features for Tec21 first-period variables
X_raw_era      = X_raw.copy()
X_test_raw_era = X_test_raw.copy()

tec21_fp_cols = [
    "average.first.period",
    "failed.subject.first.period",
    "dropped.subject.first.period",
]
for col in tec21_fp_cols:
    if col in X_raw_era.columns:
        X_raw_era[f"{col}_x_tec21"]      = X_raw_era[col].fillna(0)      * X_raw_era[ERA_COL]
        X_test_raw_era[f"{col}_x_tec21"] = X_test_raw_era[col].fillna(0) * X_test_raw_era[ERA_COL]

era_interaction_cols = [f"{c}_x_tec21" for c in tec21_fp_cols if c in X_raw_era.columns]
print(f"Era interaction features added: {era_interaction_cols}")

numeric_cols_m6 = NUM_COMMON + NUM_TEC21 + era_interaction_cols
preprocessor_m6 = build_preprocessor(numeric_cols_m6)
X_train_m6 = preprocessor_m6.fit_transform(X_raw_era)
X_test_m6  = preprocessor_m6.transform(X_test_raw_era)
print(f"M6 feature matrix: {X_train_m6.shape[1]} columns "
      f"({len(era_interaction_cols)} new interaction cols)")

class_ratio = (y_train == 0).sum() / (y_train == 1).sum()
xgb_m6 = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=class_ratio,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="aucpr",
    early_stopping_rounds=30,
    random_state=SEED,
    n_jobs=-1,
)
xgb_m6.fit(
    X_train_m6, y_train,
    eval_set=[(X_test_m6, y_test)],
    verbose=False,
)
prob_test_m6 = xgb_m6.predict_proba(X_test_m6)[:, 1]
results.append(evaluate("M6 – XGBoost + era interaction features", y_test, prob_test_m6))

In [ ]:
def build_preprocessor_with_groups(numeric_cols):
    """Same as build_preprocessor but passes school_code and era_code through as raw integers."""
    return ColumnTransformer(
        transformers=[
            ("num",    StandardScaler(),                numeric_cols),
            ("cat",    OneHotEncoder(
                           handle_unknown="infrequent_if_exist",
                           sparse_output=False,
                           min_frequency=30,
                       ),                              CAT_COMMON),
            ("flags",  "passthrough",                  flag_cols),
            ("school", "passthrough",                  [SCHOOL_COL]),
            ("era",    "passthrough",                  [ERA_COL]),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

preprocessor_m6b = build_preprocessor_with_groups(numeric_cols_m6)
X_train_m6b = preprocessor_m6b.fit_transform(X_raw_era)
X_test_m6b  = preprocessor_m6b.transform(X_test_raw_era)
print(f"M6b feature matrix: {X_train_m6b.shape[1]} columns  (M6 had {X_train_m6.shape[1]})")

xgb_m6b = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=class_ratio,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="aucpr",
    early_stopping_rounds=30,
    random_state=SEED,
    n_jobs=-1,
)
xgb_m6b.fit(
    X_train_m6b, y_train,
    eval_set=[(X_test_m6b, y_test)],
    verbose=False,
)
prob_test_m6b = xgb_m6b.predict_proba(X_test_m6b)[:, 1]
results.append(evaluate("M6b – XGBoost + era + school_code", y_test, prob_test_m6b))

### Decomposición de la varianza del modelo final

In [ ]:
alpha_samples = trace_full.posterior["alpha_school"]      # (chain, draw, school)
sigma_alpha_samples = trace_full.posterior["sigma_alpha"] # (chain, draw)

sigma_alpha_mean = float(sigma_alpha_samples.mean())
sigma_alpha_hdi  = az.hdi(trace_full, var_names=["sigma_alpha"])["sigma_alpha"].values

print("\n── Between-school variance ─────────────────────────────────────────")
print(f"  σ_α posterior mean : {sigma_alpha_mean:.3f}")
print(f"  σ_α 94% HDI        : [{sigma_alpha_hdi[0]:.3f}, {sigma_alpha_hdi[1]:.3f}]")

# School-level caterpillar plot (sorted by posterior mean)
alpha_mean = alpha_samples.mean(dim=["chain", "draw"]).values
alpha_hdi  = az.hdi(trace_full, var_names=["alpha_school"])["alpha_school"].values

school_effect_df = pd.DataFrame({
    "school":      coords_full["school"],
    "alpha_mean":  alpha_mean,
    "hdi_low":     alpha_hdi[:, 0],
    "hdi_high":    alpha_hdi[:, 1],
}).sort_values("alpha_mean")

fig, ax = plt.subplots(figsize=(8, max(4, len(coords_full["school"]) * 0.35)))
for i, row in enumerate(school_effect_df.itertuples()):
    ax.plot([row.hdi_low, row.hdi_high], [i, i], color="steelblue", lw=1.5)
    ax.plot(row.alpha_mean, i, "o", color="steelblue", ms=4)
ax.axvline(0, color="grey", lw=0.8, ls="--")
ax.set_yticks(range(len(school_effect_df)))
ax.set_yticklabels(school_effect_df["school"].tolist(), fontsize=8)
ax.set_xlabel("School random intercept α_j  (log-odds scale)")
ax.set_title("School-level random intercepts — 94% HDI\n(positive = higher dropout tendency, net of covariates)")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "school_random_intercepts.png", dpi=150)
plt.show()
print(f"\nCaterpillar plot saved → {OUTPUT_DIR / 'school_random_intercepts.png'}")

# Era fixed effect interpretation
beta_era_post = trace_full.posterior["beta_era"]
beta_era_mean = float(beta_era_post.mean())
beta_era_hdi  = az.hdi(trace_full, var_names=["beta_era"])["beta_era"].values

print(f"\n── Era (Tec21) fixed effect ────────────────────────────────────────")
print(f"  β_era posterior mean : {beta_era_mean:.3f}")
print(f"  β_era 94% HDI        : [{beta_era_hdi[0]:.3f}, {beta_era_hdi[1]:.3f}]")
print(f"  Odds ratio           : {np.exp(beta_era_mean):.3f}")
print(f"  (positive = Tec21 associated with higher dropout on log-odds scale)")

# Comparación de los modelos

In [ ]:
results_df = pd.DataFrame(results)
print("\n" + "="*70)
print("  MODEL COMPARISON — Test set  (sorted by dropout_recall @ threshold)")
print("="*70)
display(
    results_df.set_index("model")
    .sort_values("dropout_recall", ascending=False)
    [["dropout_recall", "dropout_precision", "pr_auc", "roc_auc", "brier", "threshold"]]
)

results_df.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)

# ── ROC + Precision-Recall curves ────────────────────────────────────────────
all_probs = [
    ("M1 – Baseline LR",          prob_test_m1),
    ("M2a – Era LR",               prob_test_m2a),
    ("M2b – PyMC MAP",             prob_test_m2b_map),
    ("M3 – Multilevel ADVI",      prob_test_m3),
    ("M4 – XGBoost",               prob_test_m4),
    ("M4b – LightGBM",             prob_test_m4b),
    ("M5 – SMOTE + LR",            prob_test_m5),
    ("M5b – SMOTE + XGBoost",     prob_test_m5b),
    ("M6 – XGBoost + era",         prob_test_m6),
    ("M6b – XGBoost + era + school", prob_test_m6b),
]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax_roc = axes[0]
for label, prob in all_probs:
    RocCurveDisplay.from_predictions(y_test, prob, name=label, ax=ax_roc)
ax_roc.set_title("ROC curves — test set")
ax_roc.legend(fontsize=7, loc="lower right")

ax_pr = axes[1]
for label, prob in all_probs:
    PrecisionRecallDisplay.from_predictions(y_test, prob, name=label, ax=ax_pr)
ax_pr.set_title("Precision-Recall curves — test set\n(more informative for imbalanced data)")
ax_pr.legend(fontsize=7, loc="upper right")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "roc_pr_curves.png", dpi=150)
plt.show()
print(f"ROC + PR plots saved → {OUTPUT_DIR / 'roc_pr_curves.png'}")

### NUTS

Opcional

In [ ]:
if RUN_NUTS:
    sub_idx = (
        pd.DataFrame({"school": school_train, "y": y_train, "idx": np.arange(len(y_train))})
        .groupby(["school", "y"], group_keys=False)
        .apply(lambda g: g.sample(min(len(g), 200), random_state=SEED))  # ≤200/cell
        ["idx"].values
    )
    X_sub     = X_train_m3[sub_idx]
    era_sub   = era_train[sub_idx]
    school_sub = school_train[sub_idx]
    y_sub     = y_train[sub_idx]

    print(f"NUTS subsample: {len(y_sub):,} rows  |  dropout {y_sub.mean():.3%}")

    with pm.Model(coords=coords_full) as nuts_model:
        X_data_n    = pm.Data("X",          X_sub.astype("float32"), dims=("obs", "feature"))
        era_data_n  = pm.Data("era_idx",    era_sub,                  dims="obs")
        school_data_n = pm.Data("school_idx", school_sub,             dims="obs")

        mu_alpha    = pm.Normal("mu_alpha",    0.0, 1.5)
        sigma_alpha = pm.HalfNormal("sigma_alpha", 1.0)
        z_school    = pm.Normal("z_school",    0.0, 1.0, dims="school")
        alpha_school = pm.Deterministic(
            "alpha_school", mu_alpha + sigma_alpha * z_school, dims="school"
        )
        beta_era    = pm.Normal("beta_era",  0.0, 1.0)
        beta        = pm.Normal("beta",      0.0, 1.0, dims="feature")

        logit_p = (
            alpha_school[school_data_n]
            + beta_era * era_data_n
            + pm.math.dot(X_data_n, beta)
        )
        pm.Bernoulli("y_obs", logit_p=logit_p, observed=y_sub)

    with nuts_model:
        trace_nuts = pm.sample(
            draws=1000, tune=1000, chains=4,
            target_accept=0.9,
            random_seed=SEED,
            progressbar=True,
        )

    print(az.summary(trace_nuts, var_names=["mu_alpha", "sigma_alpha", "beta_era"],
                     hdi_prob=0.94))

    trace_nuts.to_netcdf(OUTPUT_DIR / "trace_nuts_subsample.nc")
    print(f"\nNUTS trace saved → {OUTPUT_DIR / 'trace_nuts_subsample.nc'}")

## Guardar traces para análisis futuro

In [ ]:
trace_era.to_netcdf( OUTPUT_DIR / "trace_era_advi.nc")
trace_full.to_netcdf(OUTPUT_DIR / "trace_multilevel_advi.nc")
print("Traces saved.")
print("\nDone. Outputs in:", OUTPUT_DIR.resolve())